# Premier League Analytics — 01: Data Acquisition

Downloads and cleans two seasons of Premier League match data from football-data.co.uk.  
Run `python data/fetch_data.py` first.

In [1]:
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':  # Jupyter starts in notebooks/; paths are repo-relative
    os.chdir('..')

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

DATA = Path('data')
for f in [DATA / 'E0_2223.csv', DATA / 'E0_2324.csv']:
    if not f.exists():
        raise FileNotFoundError(f'{f} not found — run data/fetch_data.py')

## 1. Load Match Data

In [2]:
s2223 = pd.read_csv(DATA / 'E0_2223.csv')
s2324 = pd.read_csv(DATA / 'E0_2324.csv')

s2223['season'] = '2022-23'
s2324['season'] = '2023-24'

matches = pd.concat([s2223, s2324], ignore_index=True)
print(f'Combined: {len(matches):,} matches × {matches.shape[1]} columns')
matches.head(3)

Combined: 760 matches × 107 columns


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,season
0,E0,05/08/2022,20:00,Crystal Palace,Arsenal,0,2,A,0,1,...,0.50,2.09,1.84,2.04,1.88,2.09,1.88,2.03,1.85,2022-23
1,E0,06/08/2022,12:30,Fulham,Liverpool,2,2,D,1,0,...,1.75,1.90,2.03,1.91,2.02,2.01,2.06,1.89,1.99,2022-23
2,E0,06/08/2022,15:00,Bournemouth,Aston Villa,2,0,H,1,0,...,0.50,1.93,2.00,1.93,2.00,1.94,2.04,1.88,2.00,2022-23


## 2. Clean & Validate

In [3]:
# Parse date
matches['Date'] = pd.to_datetime(matches['Date'], dayfirst=True, errors='coerce')
print(f'Date nulls: {matches["Date"].isna().sum()}')

# Core numeric columns
core = ['FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']
present = [c for c in core if c in matches.columns]
for col in present:
    matches[col] = pd.to_numeric(matches[col], errors='coerce')

print(f'Null counts for core columns:')
matches[present].isna().sum()

Date nulls: 0
Null counts for core columns:


FTHG    0
FTAG    0
HS      0
AS      0
HST     0
AST     0
HC      0
AC      0
HY      0
AY      0
HR      0
AR      0
dtype: int64

## 3. Build Team-Level Season Summary

In [4]:
def team_stats(df: pd.DataFrame, side: str) -> pd.DataFrame:
    """Aggregate per-team stats for home or away side."""
    prefix = 'H' if side == 'home' else 'A'
    opp    = 'A' if side == 'home' else 'H'
    team_col = 'HomeTeam' if side == 'home' else 'AwayTeam'
    return (
        df.groupby(['season', team_col]).agg(
            games=('Date', 'count'),
            goals_for=(f'FT{prefix}G', 'sum'),
            goals_against=(f'FT{opp}G', 'sum'),
            shots=(f'{prefix}S', 'sum'),
            shots_on_target=(f'{prefix}ST', 'sum'),
            corners=(f'{prefix}C', 'sum'),
            yellows=(f'{prefix}Y', 'sum'),
        )
        .rename_axis(['season', 'team'])
        .reset_index()
    )

home_stats = team_stats(matches, 'home')
away_stats = team_stats(matches, 'away')

team_sum = pd.concat([home_stats, away_stats]).groupby(['season', 'team']).sum().reset_index()
team_sum['goal_diff'] = team_sum['goals_for'] - team_sum['goals_against']
team_sum['shot_accuracy'] = team_sum['shots_on_target'] / team_sum['shots'].replace(0, np.nan)

print(f'Team summary: {len(team_sum)} rows')
team_sum.head()

Team summary: 40 rows


,season,team,games,goals_for,goals_against,shots,shots_on_target,corners,yellows,goal_diff,shot_accuracy
0,2022-23,Arsenal,38,88,43,593,204,223,51,45,0.344013
1,2022-23,Aston Villa,38,51,46,431,151,165,80,5,0.350348
2,2022-23,Bournemouth,38,37,71,355,133,144,68,-34,0.374648
3,2022-23,Brentford,38,58,46,397,162,163,55,12,0.408060
4,2022-23,Brighton,38,72,53,612,231,233,59,19,0.377451


## 4. Save Processed Data

In [5]:
out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
matches.to_parquet(out_dir / 'matches_clean.parquet', index=False)
team_sum.to_parquet(out_dir / 'team_season_stats.parquet', index=False)
print('Saved matches_clean.parquet and team_season_stats.parquet to outputs/')

Saved matches_clean.parquet and team_season_stats.parquet to outputs/
